In [ ]:
pip install pyarrow==8.0.0


In [ ]:
pip install datasets

In [ ]:
pip install diffusers

In [ ]:
pip install gradio

In [ ]:
from datasets import load_dataset
from PIL import Image
import torchvision.transforms as transforms
from diffusers import StableDiffusionPipeline
import torch
import gradio as gr

In [ ]:
ds = load_dataset("tonyassi/fashion-design-images")

In [ ]:
transform = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])



In [ ]:
def preprocess_function(examples):
    images = []
    for img_path in examples['image']:
        img = Image.open(img_path).convert("RGB")
        img = transform(img)
        images.append(img)
    return {'pixel_values': images, 'text': examples['text']}


In [ ]:
# Load the model
model_id = "stabilityai/stable-diffusion-2-1"  # Adjust based on your model
device = "cuda" if torch.cuda.is_available() else "cpu"
pipeline = StableDiffusionPipeline.from_pretrained(model_id).to(device)

In [ ]:
def generate_image_from_prompt(prompt):
    # Generate an image based on the prompt
    images = pipeline(prompt).images
    return images[0]

In [ ]:
iface = gr.Interface(
    fn=generate_image_from_prompt,
    inputs=gr.Textbox(label="Enter a fashion prompt"),
    outputs=gr.Image(type="pil"),
    title="Fashion Design Image Generator",
    description="Generate stunning images of fashion designs based on detailed prompts.",
    examples=[
        ["Design a traditional bridal lehenga with gold embroidery and a matching dupatta."],
        ["Create a contemporary Indian sari with pastel shades and intricate beadwork."],
        ["Generate a royal Indian sherwani with rich embroidery and a regal turban."],
        ["Illustrate an elegant Indian wedding gown with detailed gold embellishments and a flowing skirt."]
    ]
)

In [ ]:
iface.launch()